In [3]:
import anthropic
import os

client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from environment

# A simple question about healthcare claims
message = client.messages.create(
    model='claude-sonnet-4-5',
    max_tokens=512,
    messages=[
        {
            'role': 'user',
            'content': 'What ICD-10 codes are associated with Type 2 diabetes?'
        }
    ]
)

print(message.content[0].text)

# ICD-10 Codes for Type 2 Diabetes

## Primary Code
**E11** - Type 2 diabetes mellitus

## Common Subcategories

### Without Complications
- **E11.9** - Type 2 diabetes mellitus without complications

### With Complications
- **E11.0x** - With hyperosmolarity
- **E11.1x** - With ketoacidosis
- **E11.2x** - With kidney complications
- **E11.3x** - With ophthalmic complications
- **E11.4x** - With neurological complications
- **E11.5x** - With circulatory complications
- **E11.6x** - With other specified complications
- **E11.8** - With unspecified complications

### Management-Related
- **E11.65** - With hyperglycemia
- **E11.649** - With hypoglycemia without coma
- **E11.641** - With hypoglycemia with coma

## Important Notes

1. **5th-6th character specificity**: Most E11 codes require additional characters for complete coding
2. **Multiple codes**: You may need multiple codes if complications affect different body systems
3. **Additional codes**: Often used with codes for specific ma

In [4]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Settings
from llama_index.llms.anthropic import Anthropic as LlamaAnthropic
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# Use a free local embedding model (downloads ~90MB on first run)
Settings.embed_model = HuggingFaceEmbedding(
    model_name='sentence-transformers/all-MiniLM-L6-v2'
)

# Use Claude as the language model
Settings.llm = LlamaAnthropic(model='claude-sonnet-4-5')

# Load all documents from the knowledge base folder
documents = SimpleDirectoryReader('../data/knowledge_base/').load_data()
print(f'Loaded {len(documents)} documents')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loaded 4 documents


In [5]:
# Build the vector index from the loaded documents
index = VectorStoreIndex.from_documents(documents)
print('Index built successfully')

# Save to disk so you don't rebuild it every time
index.storage_context.persist(persist_dir='../data/knowledge_base_index/')
print('Index saved to disk')

Index built successfully
Index saved to disk


In [6]:
from llama_index.core import StorageContext, load_index_from_storage

# Load the index from disk
storage_context = StorageContext.from_defaults(persist_dir='../data/knowledge_base_index/')
index = load_index_from_storage(storage_context)

# Create a query engine
query_engine = index.as_query_engine(
    similarity_top_k=3,      # retrieve top 3 most relevant chunks
    response_mode='compact', # combine retrieved chunks into a single prompt
)

# Ask the same question as before
question = 'What ICD-10 codes are associated with Type 2 diabetes?'
response = query_engine.query(question)

print('Answer:')
print(response.response)
print()
print('Source documents used:')
for node in response.source_nodes:
    print(f'  - Score: {node.score:.3f} | Text: {node.text[:100]}...')

Answer:
Type 2 diabetes is associated with ICD-10 code E11, which falls under Chapter IV: Endocrine, nutritional and metabolic diseases (E00-E89).

Source documents used:
  - Score: 0.558 | Text: ICD-10 Chapter I: Certain infectious and parasitic diseases (A00-B99)
ICD-10 Chapter II: Neoplasms -...
  - Score: 0.404 | Text: CARC Code 4: The service/equipment/drug is not covered by the plan.
CARC Code 16: Claim/service lack...
  - Score: 0.229 | Text: ...


In [8]:
test_questions = [
    'What ICD-10 codes relate to circulatory system diseases?',
    'What does CARC code 50 mean?',
    'What is the ICD chapter for mental health conditions?',
    'What are the most common reasons a healthcare claim gets denied?',
    'What CPT code range covers office visits?',
    'What is the difference between CARC code 4 and CARC code 50?',
    'What ICD codes are used for musculoskeletal conditions?',
    'What is a timely filing denial?',
    'What does prior authorization mean in claims processing?',
    'What ICD code is used for pneumonia?',
]

for q in test_questions:
    response = query_engine.query(q)
    print(f'Q: {q}')
    print(f'A: {response.response}')
    print('Sources:')
    for node in response.source_nodes:
        print(f'  - Score: {node.score:.3f} | {node.text[:80]}...')
    print('-' * 60)

Q: What ICD-10 codes relate to circulatory system diseases?
A: ICD-10 codes relating to circulatory system diseases fall under Chapter IX and range from I00-I99. This category includes conditions such as:

- Heart attack (I21)
- Heart failure (I50)
- Hypertension (I10)
Sources:
  - Score: 0.589 | ICD-10 Chapter I: Certain infectious and parasitic diseases (A00-B99)
ICD-10 Cha...
  - Score: 0.379 | CARC Code 4: The service/equipment/drug is not covered by the plan.
CARC Code 16...
  - Score: 0.267 | ...
------------------------------------------------------------
Q: What does CARC code 50 mean?
A: CARC Code 50 means that the services are non-covered because they are not deemed a medical necessity by the payer.
Sources:
  - Score: 0.477 | CARC Code 4: The service/equipment/drug is not covered by the plan.
CARC Code 16...
  - Score: 0.191 | ...
  - Score: 0.124 | ICD-10 Chapter I: Certain infectious and parasitic diseases (A00-B99)
ICD-10 Cha...
-------------------------------------------

In [9]:
# Reload documents (picks up the updated cpt_categories.txt)
documents = SimpleDirectoryReader('../data/knowledge_base/').load_data()
print(f'Loaded {len(documents)} documents')

# Rebuild the index
index = VectorStoreIndex.from_documents(documents)
print('Index rebuilt')

# Save to disk (overwrites the old index)
index.storage_context.persist(persist_dir='../data/knowledge_base_index/')
print('Index saved')

# Recreate the query engine
query_engine = index.as_query_engine(
    similarity_top_k=3,
    response_mode='compact',
)
print('Query engine ready')

Loaded 4 documents
Index rebuilt
Index saved
Query engine ready


In [10]:
retest_questions = [
    'What CPT code range covers office visits?',
    'What ICD codes are used for musculoskeletal conditions?',
    'What does prior authorization mean in claims processing?',
]

for q in retest_questions:
    response = query_engine.query(q)
    print(f'Q: {q}')
    print(f'A: {response.response}')
    print('Sources:')
    for node in response.source_nodes:
        print(f'  - Score: {node.score:.3f} | {node.text[:80]}...')
    print('-' * 60)

Q: What CPT code range covers office visits?
A: The CPT code range 99202-99215 covers office or other outpatient visits for Evaluation and Management. This range includes:

- 99202-99205: New patient office visits
- 99211-99215: Established patient office visits
Sources:
  - Score: 0.482 | CPT Code Range 99202-99215: Office or other outpatient visits (Evaluation and Ma...
  - Score: 0.363 | CARC Code 4: The service/equipment/drug is not covered by the plan.
CARC Code 16...
  - Score: 0.102 | ...
------------------------------------------------------------
Q: What ICD codes are used for musculoskeletal conditions?
A: Musculoskeletal conditions are classified under ICD-10 Chapter XIII, which covers diseases of the musculoskeletal system and uses codes in the range M00-M99. Some specific examples include:

- Osteoarthritis (M17)
- Back pain (M54)
- Rheumatoid arthritis (M05)
Sources:
  - Score: 0.413 | CPT Code Range 99202-99215: Office or other outpatient visits (Evaluation and Ma...
  -

In [11]:
from llama_index.core.node_parser import SentenceSplitter

# Two different chunking strategies
parser_default = SentenceSplitter(chunk_size=1024, chunk_overlap=50)
parser_small = SentenceSplitter(chunk_size=256, chunk_overlap=30)

nodes_default = parser_default.get_nodes_from_documents(documents)
nodes_small = parser_small.get_nodes_from_documents(documents)

print(f'Default chunking (1024 tokens): {len(nodes_default)} chunks')
print(f'Small chunking (256 tokens): {len(nodes_small)} chunks')

# Compare retrieval for the same question
question = 'What are the denial reasons related to medical necessity?'

index_default = VectorStoreIndex(nodes_default)
index_small = VectorStoreIndex(nodes_small)

resp_default = index_default.as_query_engine(similarity_top_k=2).query(question)
resp_small = index_small.as_query_engine(similarity_top_k=2).query(question)

print('\nDefault chunks answer:')
print(resp_default.response)
print('\nSmall chunks answer:')
print(resp_small.response)

Default chunking (1024 tokens): 4 chunks
Small chunking (256 tokens): 7 chunks

Default chunks answer:
Based on the information provided, denial reasons related to medical necessity include:

1. **CARC Code 50**: Non-covered services because they are not deemed a medical necessity by the payer.

2. **Medical necessity denial pattern**: This occurs when ICD (diagnosis) and CPT (procedure) codes are clinically inconsistent, meaning the diagnosis does not support the medical need for the service or procedure performed.

3. **CARC Code 151**: Payment adjustments when the payer determines that the information submitted does not support the frequency or number of services provided.

These denials typically happen when the healthcare service, treatment, or procedure cannot be justified as medically necessary based on the patient's diagnosis or when the documentation fails to demonstrate the clinical appropriateness of the care provided.

Small chunks answer:
Based on the information provided,

## Chunking Experiment — Findings

Tested two chunking strategies on the knowledge base:
- **Default (1024 tokens):** 4 chunks — each document became essentially one chunk
- **Small (256 tokens):** 7 chunks — slightly more granular but minimal difference

**Finding:** Both strategies produced nearly identical answers for the medical necessity query. 
With a small knowledge base (4 documents), chunking strategy has minimal impact on retrieval 
quality because the retriever finds the same content either way.

**When this would matter:** A larger knowledge base — for example, a 50-page payer policy 
document or the full ICD-10 code listing — would benefit from smaller chunks. Default chunking 
on a long document might retrieve an entire page when only one paragraph is relevant, lowering 
retrieval precision. Smaller chunks would isolate the specific relevant passage and produce 
higher similarity scores.

**Decision:** Keeping default 1024-token chunking for this project. Will revisit if the knowledge 
base is expanded with longer documents.